# MedQA walkthrough

A thin notebook on top of the `medqa` package: every heavy step is a CLI command, and the notebook only inspects artifacts.

Run from the repo root after `make setup-mac && make analysis && make index` (CPU is enough).

In [ ]:
import json
import os
from pathlib import Path

from IPython.display import Image, Markdown, display

if Path.cwd().name == "notebooks":
    os.chdir("..")

from medqa.config import load_config
from medqa.data.split import load_splits

cfg = load_config("configs/models/qwen3-1.7b.yaml")
splits = load_splits(cfg.data.split_dir)
{name: len(frame) for name, frame in splits.items()}

## 1. What the data looks like

In [ ]:
splits["train"].sample(5, random_state=0)

In [ ]:
print(json.loads((cfg.data.split_dir / "stats.json").read_text()))
for fig in ["answer_lengths", "top_words_answers", "answers_2grams"]:
    display(Image(f"runs/analysis/eda/{fig}.png", width=650))

## 2. Topics (TF-IDF + k-means)

In [ ]:
clusters = json.loads(Path("runs/analysis/clustering/clusters.json").read_text())
print(f"k={clusters['k']}  silhouette={clusters['silhouette']:.4f}")
for c in clusters["clusters"][:10]:
    print(c["id"], c["size"], ", ".join(c["top_terms"][:6]))
display(Image("runs/analysis/clustering/k_silhouette.png", width=650))

## 3. Retrieval: what RAG would put in the prompt

In [ ]:
from medqa.prompts import build_user_message
from medqa.retrieval.hybrid import HybridRetriever

retriever = HybridRetriever.load(cfg.data.index_dir)
row = splits["test"].iloc[0]
hits = retriever.search([row.question], k=3)[0]
print(build_user_message(row.question, [h.text for h in hits]))
print("\nReference answer:", row.answer)

## 4. Results (after GPU runs + `medqa report`)

In [ ]:
results = Path("docs/results.md")
if results.exists():
    display(Markdown(results.read_text().replace("../reports", "reports")))